# GBM From Ito's lemma
Start with the GBM SDE mentioned earlier:
$$dS_t = μS_tdt + σS_tdW_t \quad (1)$$

Now, let $X_t = ln(S_t)$ and let $S_0$ be an arbitrary positive and non-zero initial value. By Ito's Lemma, for a twice continously differentiable function $f(S_t)$, the differential $df$ is:
$$df(S_t) = f'(S_t)dS_t + \frac{1}{2}f''(S_t)(dS_t)^2 \quad (2)$$

Applying (2) to $X_t$, and knowing $(dS_t)^2 = σ^2S^2_tdt$ (the quadratic variation of $S_t$ as $dt\to0$, the computation is out of scope for this section so we will take the result for granted),
$$dX_t = \frac{dX_t}{dS_t}dS_t + \frac{1}{2}\frac{d^2X_t}{(dS_t)^2}(dS_t)^2$$

$$= \frac{1}{S_t}dS_t + \frac{-1}{2S^2_t}(σ^2S^2_tdt)$$

$$= \frac{1}{S_t}(μS_tdt + σS_tdW_t) - \frac{1}{2}σ^2dt$$

$$=μdt + σdW_t - \frac{1}{2}σ^2dt$$

$$=dt(μ - \frac{1}{2}σ^2) + σdW_t$$

Now, integrate both sides, plugging in $X_t = ln(S_t)$,

$$\int{d(ln(S_t))} = (μ - \frac{1}{2}σ^2)\int{dt} + σ\int{dW_t}$$

$$ln(S_t) = (μ - \frac{1}{2}σ^2)t + σW_t + C$$

Let $C = ln(S_0)$ and plug back in,

$$ln(S_t) = (μ - \frac{1}{2}σ^2)t + σW_t + ln(S_0)$$

$$ln(\frac{S_t}{S_0}) = (μ - \frac{1}{2}σ^2)t + σW_t$$

$$S_t = S_0\exp((μ - \frac{σ^2}{2})t + σW_t)\quad (3)$$

This is exactly Equation (2) in notebook 1, and thus we have achieved our goal of deriving GBM from its SDE using Ito's Lemma.

# Assumptions
- Drift and volatility are constant over the simulation horizon.
- Prices evolve continuously, so jumps are excluded.
- Innovations are iid.
- Under GBM, log returns are normally distributed, which implies prices are lognormally distributed.

In [97]:
# for modules to work when running this notebook, we need to add the project root to the sys.path
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [98]:
#imports
import numpy as np
import pandas as pd
import numpy.typing as npt

from data.data import get_multiple_stocks_data

In [99]:
# reuseable class for GBM simulation and risk metrics calculation
class GBMRiskEngine:
    ROLLING_WINDOWS = (20, 60)
    TIME_HORIZONS = (1, 10)
    ALPHAS = (0.95, 0.99) # the confidence levels for VaR and ES calculations
    DAILY = 252
    N_PATHS = 10_000
    T = 1.0  # Time to maturity in years for the simulation
    N = 252  # Number of time steps in the simulation (daily steps for 1 year)

    def simulate_euler_maruyama(
        self, 
        S0: float,
        mu_annual: float,
        sigma_annual: float,
        T: float,
        N: int,
        n_paths: int
    ) -> npt.NDArray[np.float64]:
      N = int(np.floor(N))
      z = np.random.standard_normal((n_paths, N))
      delta_t = T / N
      paths = np.zeros((n_paths, N + 1))
      paths[:, 0] = S0
      paths[:, 1:] = S0 * np.cumprod(1 + mu_annual * delta_t + sigma_annual * np.sqrt(delta_t) * z, axis=1)
      return paths

    def simulate_gbm(
        self,
        S0: float,
        mu_annual: float,
        sigma_annual: float,
        T: float,
        N: int,
        n_paths: int
    ) -> npt.NDArray[np.float64]:
        N = int(N)
        dt = T / N
        z = np.random.standard_normal((n_paths, N))

        log_increments = (
            (mu_annual - 0.5 * sigma_annual**2) * dt
            + sigma_annual * np.sqrt(dt) * z
        )

        paths = np.zeros((n_paths, N + 1))
        paths[:, 0] = S0
        paths[:, 1:] = S0 * np.exp(np.cumsum(log_increments, axis=1))
        return paths

    def var_es(
        self,
        paths: npt.NDArray[np.float64],
        time_horizon: int,
        alpha: float
    ) -> tuple[float, float]:
        if time_horizon < 1 or time_horizon >= paths.shape[1]:
            raise ValueError("time_horizon must be between 1 and number of simulated steps")

        P0 = paths[0, 0]
        losses = P0 - paths[:, time_horizon]

        var = np.percentile(losses, alpha * 100)
        es = losses[losses > var].mean()

        return float(var), float(es)

In [100]:
# initialize the risk engine and variables to store data
risk_eng = GBMRiskEngine()

tickers = ["SPY", "GLD", "NVDA", "GOOGL", "BTC-USD"]
data = get_multiple_stocks_data(tickers, "2016-06-01", "2026-06-01", "1d")

prices = {}
log_returns = {}
sigmas_by_window = {}
paths_by_ticker = {}
results = {}

In [101]:
# populate the dictionaries with prices, log returns, volatilities, simulated paths, and risk metrics
for ticker in tickers:
    curr_data = data[ticker]
    prices[ticker] = curr_data["Close"]

    lr = pd.Series(
        np.diff(np.log(prices[ticker].values)),
        index=prices[ticker].index[1:]
    )
    log_returns[ticker] = lr

    mu_annual = lr.mean() * risk_eng.DAILY

    sigmas_by_window[ticker] = {
        f"vol_{window}d": lr.rolling(window).std().dropna().iloc[-1] * np.sqrt(risk_eng.DAILY)
        for window in risk_eng.ROLLING_WINDOWS
    }

    paths_by_ticker[ticker] = {}
    results[ticker] = {}

    for window_key, sigma_annual in sigmas_by_window[ticker].items():
        paths = risk_eng.simulate_gbm(
            S0=prices[ticker].iloc[-1],
            mu_annual=mu_annual,
            sigma_annual=sigma_annual,
            T=risk_eng.T,
            N=risk_eng.N,
            n_paths=risk_eng.N_PATHS
        )

        paths_by_ticker[ticker][window_key] = paths
        results[ticker][window_key] = {}

        for horizon in risk_eng.TIME_HORIZONS:
            horizon_key = f"{horizon}d"
            results[ticker][window_key][horizon_key] = {}

            for alpha in risk_eng.ALPHAS:
                var, es = risk_eng.var_es(paths, horizon, alpha)
                conf = int(alpha * 100)

                results[ticker][window_key][horizon_key][f"var_{conf}"] = var
                results[ticker][window_key][horizon_key][f"es_{conf}"] = es

In [102]:
# ------------------------------------------------------------------
# 1. Flatten the nested `results` dict into a tidy long-format table
# ------------------------------------------------------------------
metric_labels = {
    "var_95": "VaR 95%",
    "es_95": "ES 95%",
    "var_99": "VaR 99%",
    "es_99": "ES 99%",
}
metrics_order = ["var_95", "es_95", "var_99", "es_99"]

rows = []
for ticker in tickers:
    for window_key in results[ticker]:
        for horizon_key in results[ticker][window_key]:
            formatted_window = window_key.replace("vol_", "").replace("d", "d Vol")
            for metric in metrics_order:
                rows.append({
                    "Ticker": ticker,
                    "Horizon": horizon_key,
                    "Metric": metric,
                    "Vol Window": formatted_window,
                    "Value": results[ticker][window_key][horizon_key][metric],
                })

comparison_table = pd.DataFrame(rows)

# ------------------------------------------------------------------
# 2. Pivot Vol Window into its own columns (same idea as before, but
#    windows go across instead of Monte Carlo / Parametric)
# ------------------------------------------------------------------
comparison_pivot = comparison_table.pivot_table(
    index=["Ticker", "Horizon", "Metric"],
    columns="Vol Window",
    values="Value"
).reset_index()

window_cols = [c for c in comparison_pivot.columns if c not in ("Ticker", "Horizon", "Metric")]

# If there are exactly two windows (e.g. vol_20d vs vol_60d), add a
# Difference column automatically, same pattern as the MC/Param table.
diff_col = None
if len(window_cols) == 2:
    diff_col = f"Difference ({window_cols[1]} - {window_cols[0]})"
    comparison_pivot[diff_col] = comparison_pivot[window_cols[1]] - comparison_pivot[window_cols[0]]

# ------------------------------------------------------------------
# 3. Enforce sensible ordering (numeric horizon, fixed metric order)
# ------------------------------------------------------------------
horizon_order = sorted(
    comparison_pivot["Horizon"].unique(),
    key=lambda x: int(x.replace("d", ""))
)
metric_order = [metric_labels[m] for m in metrics_order]

comparison_pivot["Metric"] = comparison_pivot["Metric"].map(metric_labels)
comparison_pivot["Horizon"] = pd.Categorical(
    comparison_pivot["Horizon"], categories=horizon_order, ordered=True
)
comparison_pivot["Metric"] = pd.Categorical(
    comparison_pivot["Metric"], categories=metric_order, ordered=True
)

comparison_pivot = comparison_pivot.sort_values(
    ["Ticker", "Horizon", "Metric"]
).reset_index(drop=True)

comparison_grouped = comparison_pivot.set_index(["Ticker", "Horizon", "Metric"])

# ------------------------------------------------------------------
# 4. Detect group boundaries so we know where to draw borders
# ------------------------------------------------------------------
idx = comparison_grouped.index
ticker_vals = idx.get_level_values("Ticker").to_numpy()
horizon_vals = idx.get_level_values("Horizon").to_numpy()
n = len(idx)

new_ticker = np.zeros(n, dtype=bool)
new_horizon = np.zeros(n, dtype=bool)
for i in range(1, n):
    new_ticker[i] = ticker_vals[i] != ticker_vals[i - 1]
    new_horizon[i] = new_ticker[i] or (horizon_vals[i] != horizon_vals[i - 1])

TICKER_BORDER = "border-top: 3px solid #4695f0;"   # thick dark line between tickers
HORIZON_BORDER = "border-top: 1px solid #b0b7bd;"

def body_border_func():
    counter = {"i": 0}
    def func(row):
        i = counter["i"]
        counter["i"] += 1
        css = TICKER_BORDER if new_ticker[i] else (HORIZON_BORDER if new_horizon[i] else "")
        return [css] * len(row)
    return func

def index_border_func(level_changed_array, border_css):
    def func(s):
        return [border_css if level_changed_array[i] else "" for i in range(len(s))]
    return func

# ------------------------------------------------------------------
# 5. Style: number formatting, diff color gradient, header, borders,
#    and a hover effect that also darkens the gradient-colored cells
# ------------------------------------------------------------------
CMAP = "BrBG_r"

fmt = {c: "{:.4f}" for c in window_cols}
if diff_col:
    fmt[diff_col] = "{:+.4f}"

diff_vals = comparison_grouped[diff_col]

styled = (
    comparison_grouped.style
    .format(fmt)
    .background_gradient(
        subset=[diff_col] if diff_col else [],
        cmap=CMAP,          # note: NOT coolwarm_r, see below
        gmap=diff_vals.abs(),     # color driven by |difference|, not the signed value
        vmin=0,                  # zero deviation -> low end of cmap
        vmax=diff_vals.abs().max()  # biggest deviation -> high end
    )
    .set_properties(**{"text-align": "center"})
    .apply(body_border_func(), axis=1)
    .apply_index(index_border_func(new_horizon, HORIZON_BORDER), axis=0, level=1)
    .apply_index(index_border_func(new_horizon, HORIZON_BORDER), axis=0, level=2)
    .apply_index(index_border_func(new_ticker, TICKER_BORDER), axis=0, level=0)
    .apply_index(index_border_func(new_ticker, TICKER_BORDER), axis=0, level=1)
    .apply_index(index_border_func(new_ticker, TICKER_BORDER), axis=0, level=2)
    .set_table_styles([
        {"selector": "th", "props": [
            ("text-align", "center"),
            ("background-color", "#2c3e50"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("padding", "6px 10px"),
        ]},
        {"selector": "td", "props": [("padding", "6px 10px")]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("font-family", "Arial, sans-serif"),
            ("font-size", "13px"),
        ]},
        {"selector": "tbody tr", "props": [("transition", "background-color 0.3s ease")]},
                {"selector": "tbody tr:hover", "props": [("background-color", "#494a4b")]},
        {"selector": "tbody tr td", "props": [("transition", "box-shadow 0.3s ease")]},
        {"selector": "tbody tr:hover td", "props": [
            ("box-shadow", "inset 0 0 0 9999px rgba(0,0,0,0.12)")
        ]},
        {"selector": "caption", "props": [
            ("font-size", "14px"),
            ("font-weight", "bold"),
            ("padding", "8px 0"),
        ]},
    ], overwrite=False)
    .set_caption("Volatility Window Comparison — VaR / ES by Window")
)

styled  # merged Ticker/Horizon groups, borders, gradient-aware hover

# Model limitations
Going off the previously mentioned assumptions: 

- Under GBM, prices are lognormally distributed and log returns are normally distributed with iid increments. This is the distributional assumption behind the Monte Carlo VaR baseline, but Notebook 01 shows that real market returns are not normal.
- GBM assumes continuous price paths, so it cannot capture jumps or discontinuities.
- GBM assumes constant volatility, while real markets exhibit time-varying volatility, clustering, and, for many assets, skew/smile effects.

# Short interpretation of results
We observe two major trends: an increase in either the length of the rolling window with which we calculate volatility or the time horizon both lead to an increase in VaR and thus also in ES. The latter effect is obvious, the later we consider terminal returns, the higher the probability the asset has experienced a bigger loss at the terminal time (the effect of volatility is larger), i.e. there were more opportunities for adverse moves. The former effect is more interesting, it seems that changing the rolling window changes the volatility estimate itself: shorter windows react faster but are noisier, while longer windows are smoother but can lag recent shifts in market conditions.

## Additional notes
The analytical GBM solution is used in the simulations above, and thus all results use the analytical solution. However, I have written the Euler-Maruyama discretization (i.e. a numerical solution to GBM) aswell for learning/reference (it required some handwritten derivation to implement in code!) and potential future use (e.g. comparison of numerical vs. analytical).